# Lecture 19: Asking for one answer, automatically

**The challenge.** Lecture 18 found, by working through the necessary
conditions by hand, that two axes moving in minimum time have a whole family
of optimal motions, and that shooting breaks down on them. A solver should
not need a person to notice that. This lecture shows that the problem
*statement* already says when the answer will not be unique, how the package
reads it, and how a **policy** then picks one motion from the family. Then
it puts the pieces to work on the problem the last few lectures have been
heading for: a two-link arm moving as fast as it can, with bounded jerks and
torque limits on both joints.

**After this lecture you can**

- tell from a problem statement whether its optimal motion is unique, by
  splitting it into the parts that nothing connects;
- explain why a limit that couples two parts does not make the motion unique;
- choose a policy (a secondary cost, a two-stage solve, or none) and say
  what each returns;
- describe how a boundary arc keeps bounded controls within their bounds
  when riding a limit leaves a line of possible controls.

## 1. Reading the statement

Lecture 18's argument used one property of the two axes: nothing connects
them except the shared final time. The Hamiltonian is a sum of one part per
axis,

$$
H = 1 + H_1(x_1, \lambda_1, u_1) + H_2(x_2, \lambda_2, u_2),
$$

each part's costates evolve on their own, and the end conditions concern
one axis each. Setting the slower axis's costates to zero then satisfies
every condition, whatever that axis does, so long as it gets there in time.

That property can be checked without solving anything. Put every state and
control in a graph and join two of them when

- one appears in the other's dynamics,
- both appear in one term of the cost, or in one terminal constraint, or
- both are components of one unit-vector control.

The connected pieces are the problem's **components**. A component is
**priced** when some cost term involves one of its variables. The rule:

> The optimal motion is not unique when some component that has controls is
> priced by no cost term, unless it is the only such component and the cost
> is a free final time.

A single axis in minimum time is the exception, and it is fine: its own
motion sets the time, and Lecture 11 solved it exactly. Two axes in minimum
time break the rule, because only one of them can set the time. A
fixed-time problem with an axis that no cost term mentions breaks it too:
any feasible motion of that axis is optimal. `policy.analyze` applies the
rule.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control import examples
from auto_optimal_control.homotopy import Homotopy
from auto_optimal_control.policy import IllPosedProblem, Lexicographic, SecondaryCost, analyze

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

for name in ("jerk_limited_move", "two_axes_min_time", "two_link_arm", "two_link_arm_min_time"):
    a = analyze(getattr(examples, name)())
    parts = "; ".join(", ".join(map(str, c.controls)) for c in a.controlled)
    print(f"{name:22s} controlled parts: {parts:8s} priced: "
          f"{[c.priced for c in a.controlled]}  unique: {not a.degenerate}")

The S-curve of Lecture 15 is one component in minimum time: unique. Lecture
18's two axes are two components, neither priced: not unique. Lecture 14's
arm also has two components, one per joint (more on that below), but its
cost, the jerk energy $\int (j_1^2 + j_2^2)/2$, has a term on each, so both
are priced: unique. The same arm in minimum time is not. Every other example
and benchmark in the package passes the check, as the test suite confirms.

The rule asks only for the structure of the expressions, so it is cheap and
it is exact about what it sees. What it cannot see is a problem that is
connected but nearly flat in some direction for numerical reasons; that
shows up only as a badly conditioned solve.

## 2. Limits do not connect

Why does the arm have two components? Its dynamics are two separate chains
of integrators, $\dot q_i = \omega_i$, $\dot\omega_i = a_i$, $\dot a_i = j_i$:
the joints are coupled only through the torques,

$$
\tau = M(q)\,a + c(q, \omega),
$$

which appear only in the torque limits $|\tau_i| \le \tau_{i,\max}$. A path
constraint is not counted as a connection, because it connects its
variables only while it is active. On a free arc it adds nothing to the
necessary conditions, and on the free arcs between the limits the argument
of Lecture 18 applies unchanged: the joint that is not setting the pace there
has room to move in many ways. `analyze` says so.

In [ ]:
print(analyze(examples.two_link_arm_min_time()))

## 3. Policies

A problem whose motion is not unique needs a rule for which motion to
return: a **policy**. `Homotopy` takes one as `policy=`:

- **`SecondaryCost(weight=1e-2)`**, the default. It adds
  $w \int \sum_i (u_i / s_i)^2 / 2$ over the controls of the unpriced
  components, with $s_i$ each control's bound, so the weight is relative to
  the time cost. This is Lecture 18's formulation: well-posed for every
  $w > 0$, selecting the smoothest of the fastest motions, at the price of a
  final time slightly above the minimum and a conditioning that grows like
  $1/w$.
- **`Lexicographic(delta=1e-2)`**, the two-stage solve of Lecture 18's
  section 5. When no limit links the components, each one's minimum time
  is a well-posed problem of its own: solve each, take the largest, $T^*$,
  then fix $t_f = (1 + \delta) T^*$ and minimize the effort.
- **`"as_stated"`** solves the problem as given, and **`"strict"`** raises
  an error instead.

With the default, `"auto"`, the package applies `SecondaryCost` when
`analyze` finds the problem degenerate and warns that it did; on a unique
problem it changes nothing. The warning is the point: the answer now depends
on a choice the user may want to make differently.

In [ ]:
try:
    Homotopy(examples.two_axes_min_time(), policy="strict")
except IllPosedProblem as err:
    print("strict:", str(err).splitlines()[-1])

P2 = [1.0, 10.0, 2.0]                      # J, D1, D2: Lecture 18's two axes
T1 = (32 * P2[1] / P2[0]) ** (1 / 3)        # axis 1 alone: 4 (D1 / 2J)^(1/3)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    h_auto = Homotopy(examples.two_axes_min_time())
print("warned:", str(caught[0].message).splitlines()[-1])
r_auto = h_auto.solve(P2, ([-3] * 6, [3] * 6), (2.0, 20.0))
h_lex = Homotopy(examples.two_axes_min_time(), policy=Lexicographic(0.01))
r_lex = h_lex.solve(P2, ([-3] * 6, [3] * 6), (2.0, 20.0))


def quintic_gap(r):
    tr = r.shooting.trajectory(r.solution.v, n_points=2001)
    s = tr["t"] / tr["t"][-1]
    return np.abs(tr["z"][:, 3] - P2[2] * (10 * s**3 - 15 * s**4 + 6 * s**5)).max()


print(f"\n{'policy':22s} {'t_f':>10s} {'t_f - T1':>9s}  axis 2 - quintic")
for label, r in (("SecondaryCost(0.01)", r_auto), ("Lexicographic(0.01)", r_lex)):
    print(f"{label:22s} {r.solution.tf:10.6f} {r.solution.tf - T1:9.1e}  {quintic_gap(r):.1e}")
print("stage 1 of the lexicographic solve, each axis alone:",
      [round(s.solution.tf, 10) for s in r_lex.stages])

Both policies return the same kind of motion. In each, axis 2 is the
minimum-jerk quintic of Lecture 14 for the time it is given, to rounding:
among the fastest motions, both pick the smoothest. They differ in the
time, and in who chooses it. The secondary cost finishes
$1.7\times10^{-4}$ above $T_1$, an amount set by the weight rather than
chosen. The lexicographic solve finds $T_1$ itself, to every digit, in its
first stage, and then runs at the margin $\delta$ the user asked for, here
1%. That margin also gives axis 1 room to be smooth. The lexicographic
policy is the more precise of the two and the more restricted: its first
stage needs components that no limit links, so it does not apply to the
arm.

## 4. Riding a limit with bounded controls

Before the arm can be solved in minimum time, one gap in the boundary arcs
of Lecture 14 has to close. There, the jerks were unbounded. Riding one
torque limit fixes one combination of them,

$$
\dot\tau_1 = M_{11}(q)\, j_1 + M_{12}(q)\, j_2 + (\text{terms in } q, \omega, a) = 0,
$$

and $H$ picks the other combination. Now each jerk is bounded,
$|j_i| \le j_{\max}$. Riding the limit leaves a *line* of jerk pairs, and
the bounds cut it to a segment. With $H$ quadratic in the jerks (the
secondary cost makes it so), the best point of the segment is the best
point of the whole line, clipped to the segment: one parameter along the
line, clipped to an interval.

In [ ]:
# Riding a torque limit with two bounded jerks: the limit leaves a line of jerk pairs, the
# bounds cut it to a segment, and the best pair is the best point of that segment.
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.6))
a1, a2 = 2.2, 0.8                       # how each jerk moves the torque (illustrative)
for ax, (c, centre, title, off) in zip(axes, [
        (0.3, (-0.35, 0.9), "the best pair on the line is inside the square", None),
        (-1.6, (1.3, 1.6), "it is outside: clip along the line to the square", (-80, 20))]):
    ax.add_patch(plt.Rectangle((-1, -1), 2, 2, fill=False, lw=1.5, color="k"))
    j1 = np.linspace(-1.5, 1.5, 400)
    j2 = -(c + a1 * j1) / a2           # a1 j1 + a2 j2 + c = 0: d(tau)/dt = 0
    ax.plot(j1, j2, color=GRAY, lw=1.2, ls="--")
    inside = (np.abs(j1) <= 1) & (np.abs(j2) <= 1)
    ax.plot(j1[inside], j2[inside], color=BLUE, lw=4, solid_capstyle="round",
            label="pairs that ride the limit within the bounds")
    for r in (0.3, 0.6, 0.9, 1.2, 1.5):  # level sets of the effort part of H
        ax.add_patch(plt.Circle(centre, r, fill=False, color=GRAY, lw=0.6, alpha=0.6))
    ax.plot(*centre, "+", color="k", ms=10, mew=1.5)
    ax.annotate("H's minimum,\nlimit ignored", centre, xytext=(8, 8), textcoords="offset points",
                fontsize=8)
    # The best point on the whole line: the projection of the centre onto it.
    n = np.array([a1, a2]) / np.hypot(a1, a2)
    star = np.array(centre) - (n @ np.array(centre) + c / np.hypot(a1, a2)) * n
    ax.plot(*star, "o", color=GRAY, ms=7, mfc="white", mew=1.5)
    if off is not None:                # in the left panel it is the riding point itself
        ax.annotate("best on the line", star, xytext=off, textcoords="offset points",
                    fontsize=8, arrowprops=dict(arrowstyle="-", color=GRAY))
    seg = np.column_stack([j1[inside], j2[inside]])
    best = seg[np.argmin(((seg - star) ** 2).sum(axis=1))]
    ax.plot(*best, "o", color=ORANGE, ms=9)
    label = "the riding jerks" + ("\n(the best on the line)" if off is None else "")
    ax.annotate(label, best, xytext=(45, -40) if off is None else (-30, -48),
                textcoords="offset points",
                fontsize=9, color=ORANGE, arrowprops=dict(arrowstyle="-", color=ORANGE))
    ax.set(xlim=(-1.6, 1.9), ylim=(-1.6, 2.1), xlabel="$j_1$", ylabel="$j_2$")
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=10, loc="left")
    ax.grid(alpha=0.2)
axes[0].legend(frameon=False, fontsize=8, loc="lower left")
fig.tight_layout()

`derive` now builds that law in closed form whenever riding leaves exactly
a line (one more control than limits ridden). One control is kept as the
line's parameter $s$, the others are solved from the limits as affine
functions of it, and each bound becomes an interval of $s$. The multiplier
$\mu$ of the limit comes from the stationarity row of a control that is off
its bound,

$$
\frac{\partial H}{\partial j_k} + \mu\,\frac{\partial \dot\tau_1}{\partial j_k} = 0,
$$

since the row of a control at its bound also carries that bound's own
multiplier. So $\mu$ changes formula where the clip does. The
Karush–Kuhn–Tucker conditions hold at every point: the test suite checks
them at thousands of random states.

One more change follows. At the junctions of an order-1 limit, Lecture 14
replaced $H$ continuity with $\dot g = 0$ at the entry and $\mu = 0$ at the
exit: simple roots where $H$ continuity has a double one, so junction times
come out exact. That rests on the control being continuous at the junction
(Jacobson, Lele and Speyer, 1971), which holds when it is strictly inside
its bounds there. A control still at its bound arrives with a jump instead.
Shooting now chooses per junction, from the control at the junction: the
exact rows when it is inside, $H$ continuity when it is not.

## 5. The arm in minimum time

`examples.two_link_arm_min_time()` is the arm of Lecture 14 with $t_f$ free,
the cost $t_f$ alone, $|j_i| \le j_{\max} = 1$ and the torque limits.
Nothing else is given: the default policy adds
$0.01 \int ((j_1/j_{\max})^2 + (j_2/j_{\max})^2)/2$, and the automatic
homotopy finds the arcs. First with torque limits too loose to matter:

In [ ]:
base = [1.0, 1.0, 1.0, 1.0, 1.5, 1.0, 1.0]   # m1, m2, l1, l2, q1f, q2f, j_max
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    h_arm = Homotopy(examples.two_link_arm_min_time())
print(h_arm.policy_text)


def arm(tau):
    p = base + list(tau)
    r = h_arm.solve(p, ([-3] * 6, [3] * 6), (1.0, 20.0))
    tr = r.shooting.trajectory(r.solution.v, n_points=4001)
    t1, t2 = examples.two_link_arm_torques(*p[:4], *tr["z"][:, 1:6].T)
    return p, r, tr, t1, t2


pa, ra, tra, tau1a, tau2a = arm((100.0, 100.0))
at = np.abs(np.abs(tra["u"]) - 1) < 1e-9
print(f"t_f = {ra.solution.tf:.6f}; joint 1 alone, jerk-limited: "
      f"(32 q1f / j_max)^(1/3) = {(32 * 1.5) ** (1 / 3):.6f}")
print(f"time at the jerk bound: joint 1 {at[:, 0].mean():.1%}, joint 2 {at[:, 1].mean():.1%}")
print(f"costates of joint 2 at t = 0: {np.round(ra.solution.lam0[[1, 3, 5]], 4)}")
print(f"peak torques {np.abs(tau1a).max():.3f}, {np.abs(tau2a).max():.3f}")

This is Lecture 18 again, now on an arm. Joint 1 has the longer move and
sets the time: it is at its jerk bound almost throughout, the bang-bang
profile of a jerk-limited axis, and $t_f$ is within $10^{-4}$ of that
profile's time. Joint 2 has time to spare and moves smoothly, at its bound
only briefly, and its costates are of the order of the weight $w = 0.01$,
the signature of a part of the problem that the secondary cost alone keeps
determined.

Now limit the torques below those peaks: $|\tau_1| \le 1.5$ and
$|\tau_2| \le 0.4$.

In [ ]:
pb, rb, trb, tau1b, tau2b = arm((1.5, 0.4))
print(f"t_f = {rb.solution.tf:.6f}  (exact: {rb.exact}; {rb.message})")
for kind, name in rb.arcs:
    print(f"  {kind:5s} {name}")
print(f"peak |tau1| = {np.abs(tau1b).max():.10f}, peak |tau2| = {np.abs(tau2b).max():.10f}, "
      f"peak |j| = {np.abs(trb['u']).max():.12f}")

In [ ]:
# The arm itself, every 0.25 s: shoulder at the origin, upper arm and forearm as bars. A link
# is drawn orange while its joint rides its torque limit.
l1, l2 = pb[2], pb[3]
q1t, q2t = trb["z"][:, 0], trb["z"][:, 1]
elbow = np.stack([l1 * np.cos(q1t), l1 * np.sin(q1t)], 1)
hand = elbow + np.stack([l2 * np.cos(q1t + q2t), l2 * np.sin(q1t + q2t)], 1)
on1 = np.abs(np.abs(tau1b) - pb[7]) < 1e-6
on2 = np.abs(np.abs(tau2b) - pb[8]) < 1e-6
fig, (ax, key) = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw=dict(width_ratios=[1.25, 1]))
picks = np.searchsorted(trb["t"], np.arange(0, trb["t"][-1] + 1e-9, 0.25))
picks = np.minimum(picks, len(trb["t"]) - 1)
for n, i in enumerate(picks):
    fade = 0.15 + 0.85 * n / (len(picks) - 1)
    ax.plot([0, elbow[i, 0]], [0, elbow[i, 1]], color=ORANGE if on1[i] else BLUE, lw=4,
            alpha=fade, solid_capstyle="round")
    ax.plot([elbow[i, 0], hand[i, 0]], [elbow[i, 1], hand[i, 1]],
            color=ORANGE if on2[i] else GRAY, lw=3, alpha=fade, solid_capstyle="round")
    ax.plot(*elbow[i], "o", color="k", ms=4, alpha=fade)
ax.plot(*hand.T, color="k", lw=0.8, ls=":")
ax.plot(0, 0, "o", color="k", ms=10)
ax.text(hand[0, 0], hand[0, 1] - 0.12, "start", fontsize=9, ha="center", va="top")
ax.text(hand[-1, 0], hand[-1, 1] + 0.08, "end", fontsize=9, ha="center", va="bottom")
ax.set_aspect("equal")
ax.set(xlabel="x", ylabel="y", xlim=(-1.0, 2.3), ylim=(-0.35, 1.9))
ax.set_title("the arm every 0.25 s (fainter = earlier);\norange: that joint on its torque limit",
             fontsize=10, loc="left")
ax.grid(alpha=0.2)
for name, on, row, color in [("shoulder on its limit", on1, 1, BLUE),
                             ("elbow on its limit", on2, 0, GRAY)]:
    t_ = trb["t"]
    key.fill_between(t_, row + 0.1, row + 0.9, where=on, color=ORANGE, alpha=0.8, lw=0)
    key.fill_between(t_, row + 0.1, row + 0.9, where=~on, color=color, alpha=0.25, lw=0)
    key.text(-0.02, row + 0.5, name, ha="right", va="center", fontsize=9)
both = on1 & on2
key.fill_between(trb["t"], -0.5, -0.1, where=both, color="k", alpha=0.7, lw=0)
key.text(-0.02, -0.3, "both at once", ha="right", va="center", fontsize=9)
key.set(xlim=(0, trb["t"][-1]), ylim=(-0.6, 2.0), xlabel="t", yticks=[])
for side in ("left", "right", "top"):
    key.spines[side].set_visible(False)
key.set_title("which joints ride their limits, and when", fontsize=10, loc="left")
fig.tight_layout()

In [ ]:
# The arm, moving: press play. A link turns orange while its joint rides its torque limit.
from IPython.display import HTML
from matplotlib import animation

plt.rcParams["animation.embed_limit"] = 30
l1, l2 = pb[2], pb[3]
q1t, q2t = trb["z"][:, 0], trb["z"][:, 1]
elbow = np.stack([l1 * np.cos(q1t), l1 * np.sin(q1t)], 1)
hand = elbow + np.stack([l2 * np.cos(q1t + q2t), l2 * np.sin(q1t + q2t)], 1)
on1 = np.abs(np.abs(tau1b) - pb[7]) < 1e-6
on2 = np.abs(np.abs(tau2b) - pb[8]) < 1e-6
n_frames = 60
idx = np.linspace(0, len(trb["t"]) - 1, n_frames).astype(int)
fig, ax = plt.subplots(figsize=(5, 4.2), dpi=72)
ax.plot(*hand.T, color="k", lw=0.6, ls=":")
ax.plot(0, 0, "o", color="k", ms=9)
upper, = ax.plot([], [], lw=6, solid_capstyle="round")
fore, = ax.plot([], [], lw=5, solid_capstyle="round")
joint, = ax.plot([], [], "o", color="k", ms=5)
label = ax.text(-0.6, 2.05, "", fontsize=9)
ax.set(xlim=(-0.7, 2.2), ylim=(-0.3, 2.3), xticks=[], yticks=[])
ax.set_aspect("equal")


def draw(k):
    i = idx[k]
    upper.set_data([0, elbow[i, 0]], [0, elbow[i, 1]])
    upper.set_color(ORANGE if on1[i] else BLUE)
    fore.set_data([elbow[i, 0], hand[i, 0]], [elbow[i, 1], hand[i, 1]])
    fore.set_color(ORANGE if on2[i] else GRAY)
    joint.set_data([elbow[i, 0]], [elbow[i, 1]])
    label.set_text(f"t = {trb['t'][i]:.2f}   |tau1| = {abs(tau1b[i]):5.2f}   |tau2| = {abs(tau2b[i]):4.2f}")
    return upper, fore, joint, label


anim = animation.FuncAnimation(fig, draw, frames=n_frames, interval=90, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

The move takes 15% longer. Joint 2 rides its upper torque limit while it
accelerates; later the shoulder and then the elbow brake on their lower
limits, the two arcs overlapping, and the elbow leaves its limit only after
the shoulder does. The torques reach their limits to $10^{-10}$ and never
pass them by more than that, and the jerks stay within $|j| \le 1$. An independent SciPy
simulation of the arm under this jerk history (in the benchmarks) gives the
same cost to $10^{-9}$.

The jerks and torques show how the pieces fit:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True)
t = trb["t"]
on = trb["active"]
for c, name in enumerate(rb.shooting.kernels.CONSTRAINT_NAMES):
    if on[:, c].any():
        for ax in axes:
            ax.fill_between(t, 0, 1, where=on[:, c], transform=ax.get_xaxis_transform(),
                            color=BLUE if name.startswith("tau1") else ORANGE, alpha=0.12, lw=0)
axes[0].plot(t, trb["u"][:, 0], color=BLUE, lw=1.6, label="$j_1$ (shoulder)")
axes[0].plot(t, trb["u"][:, 1], color=ORANGE, lw=1.6, label="$j_2$ (elbow)")
axes[0].set(ylabel="jerk", ylim=(-1.15, 1.15))
axes[1].plot(t, tau1b / pb[7], color=BLUE, lw=1.6, label=r"$\tau_1 / \tau_{1,\max}$")
axes[1].plot(t, tau2b / pb[8], color=ORANGE, lw=1.6, label=r"$\tau_2 / \tau_{2,\max}$")
axes[1].set(ylabel="torque / limit", xlabel="t", ylim=(-1.15, 1.15))
for ax in axes:
    ax.axhline(1, color="k", lw=0.6, ls="--")
    ax.axhline(-1, color="k", lw=0.6, ls="--")
    ax.legend(frameon=False, fontsize=8, loc="lower right", bbox_to_anchor=(1.0, 1.0), ncol=2)
    ax.grid(alpha=0.25)
axes[0].set_title("shaded: a torque limit ridden\n(blue the shoulder's, orange the elbow's)",
                  fontsize=10, loc="left")
fig.tight_layout()
ridden = on.any(axis=1)
both_ok = ridden & (np.abs(np.abs(trb["u"]) - 1) < 1e-9).any(axis=1)
print(f"on the arcs, a jerk sits at its bound {both_ok.sum() / ridden.sum():.0%} of the time")

On the free arcs the jerks are mostly at $\pm 1$: a nearly bang-bang,
nearly minimum-time motion, smoothed only at the switches by the small
secondary cost. On the arcs, the riding limit holds the torque on the
dashed line while the jerks move along the segment of section 4, sometimes
inside it and sometimes clipped to its end with one jerk at its bound, as
at the start of the elbow's arc. There both jerks jump: they are at $+1$
when $\tau_2$ reaches its limit, and riding it needs $j_2 = -1$ at once.
This is the S-curve's jump onto its acceleration limit (Lecture 15), now
with two controls. It is also the case from section 4 where a control is at
its bound at the junction, so shooting keeps $H$ continuity there.

## 6. What changed, and what did not

Nothing about this solve was written for the arm. The steps were:

1. `analyze` read the statement and found two joints that only the final
   time and the torque limits connect;
2. the default policy added a 1% jerk cost, and said so;
3. `derive` built boundary arcs that ride each torque limit with clipped
   jerks, and pairs of limits ridden at once (Lecture 14);
4. the active-set route of Lecture 15 tightened the limits from the
   unconstrained solution and grew the arcs where they were first violated;
5. the exact solve was accepted, with junction times exact where the jerks
   are inside their bounds.

What did not change is the price of the policy. $t_f$ is slightly above the
true minimum, by an amount that falls with $w$, and the conditioning grows
as $w$ falls. For a robot the jerk cost is often welcome in its own right,
because it is what keeps the slack joint from moving abruptly; for a
benchmark of pure minimum time it is a known, controlled error. The choice
is now explicit, visible in `r.policy`, and the user's to change.

## Summary

- Whether the optimal motion is unique can be read from the statement:
  split the variables into components that no dynamics, cost term or end
  condition connects. A controlled component in no cost term has a family
  of optimal motions, unless it alone sets a free final time.
- Path constraints do not count as connections: they couple their
  variables only while they are active.
- A policy picks one motion: a secondary cost (the default, with a warning),
  a lexicographic two-stage solve when the components are fully separate, or
  none.
- When riding a limit leaves a line of bounded controls, the boundary arc
  keeps them within their bounds by clipping the line's parameter, and the
  multiplier comes from the rows of the controls off their bounds.
- The two-link arm with bounded jerks and torque limits now solves in
  minimum time from random costates, its torques exactly at their limits.

## Exercises

1. Write a fixed-time problem with two axes in which the cost mentions only
   axis 1. What does `analyze` report? Solve it with `SecondaryCost` and
   describe axis 2's motion.
2. Add a running cost $(x_1 - x_2)^2$ to the two axes of section 3. Why does
   `analyze` now find one component, and is the minimum-time motion unique?
3. Solve the arm of section 5 with `SecondaryCost(0.1)` and `SecondaryCost(0.001)`
   (each compiles its own kernels). How do $t_f$, the time joint 2 spends at
   its jerk bound, and the junction times change?
4. Lexicographic does not apply to the arm. Sketch a two-stage solve that
   would: what would stage 1 have to compute, and why is it not a problem
   per joint?

## Further reading

- D. H. Jacobson, M. M. Lele and J. L. Speyer, "New necessary conditions of
  optimality for control problems with state-variable inequality
  constraints", *Journal of Mathematical Analysis and Applications* 35(2),
  1971: the continuity of the control at junctions of first-order limits.
- R. F. Hartl, S. P. Sethi and R. G. Vickson, "A survey of the maximum
  principles for optimal control problems with state constraints", *SIAM
  Review* 37(2), 1995.
- Z. Shiller, "Time-energy optimal control of articulated systems with
  geometric path constraints", *Journal of Dynamic Systems, Measurement, and
  Control* 118(1), 1996: minimum time plus a weighted effort, on a robot
  arm.
- M. E. Kahn and B. Roth, "The near-minimum-time control of open-loop
  articulated kinematic chains", *Journal of Dynamic Systems, Measurement,
  and Control* 93(3), 1971.